[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_1_Colab_flourishes.ipynb)

# 🧬 Practical 1: SNP Heritability using LD Score Regression (LDSC)

Welcome to the first practical! Here, we will learn how to estimate the SNP heritability ($h^2_{\text{SNP}}$) of a trait using LD Score Regression (LDSC).


---
## 🛠️ 1. Environment Setup & Package Installation

This Colab notebook uses a **Python runtime**, but since the LDSC tools we are using are built in R, we will execute our code using **R cell magic** (`%%R`). We first need to load this extension and install the required packages from CRAN and GitHub.

Packages used in this session:
* `data.table` & `dplyr`: For fast and efficient data manipulation.
* `remotes`: To install packages directly from GitHub.
* `GenomicSEM`: An R package that includes functions to run LDSC.


In [ ]:
# System dependencies that are often useful for R package installation
# (safe to run even if some packages are already present)
!apt-get update -qq
!apt-get install -y -qq libcurl4-openssl-dev libssl-dev libxml2-dev


In [ ]:
# Load the R magic extension to run R code in this Python notebook
%load_ext rpy2.ipython


In [ ]:
%%R
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes"),
  repos = "https://cloud.r-project.org",
  quiet=TRUE
)

# Install GenomicSEM from GitHub
# GenomicSEM is maintained on GitHub and its README recommends installation from GitHub.
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never")


In [ ]:
%%R
dir("/content/LDSC_Practical_1")
setwd("/content/LDSC_Practical_1")


### ✅ Load Required Packages

If the installation cell ran successfully, let's load our libraries!


In [ ]:
%%R
suppressWarnings(suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(GenomicSEM)
}))


---
## 📚 2. Understanding LDSC Basics

We are using the `GenomicSEM` library in R to run LDSC. There are **two primary steps** involved in running an LDSC analysis:

1.   **Munge the summary statistics:** `munge()`
     * *To "munge" means to convert raw data from one format to another, cleaning and standardizing it.*
2.   **Run LD Score Regression:** `ldsc()`
     * *Using the cleaned data to estimate heritability and assess confounding.*


### 📝 Required Information in Summary Statistics
The input summary statistics file for the `munge()` function must contain at least the following five pieces of information:

1.   **SNP ID:** The rsID of the SNP.
2.   **A1:** The effect allele.
3.   **A2:** The other allele.
4.   **Z-score:** The direction (sign) and magnitude of the association (or effect size and standard error, or $P$-value and effect direction).
5.   **N:** The sample size for the SNP.


### ⚙️ Arguments for the `munge()` Function
The `munge()` function takes several key arguments:
1. `files`: The path/name of the summary statistics file.
2. `hm3`: The path/name of the reference file (e.g., HapMap 3 SNP list).
3. `trait.names`: A string specifying the output prefix/name.
4. `info.filter`: An imputation quality score cutoff (e.g., `0.90`).
5. `maf.filter`: A minor allele frequency cutoff (e.g., `0.01`).


### ⚙️ Arguments for the `ldsc()` Function
The `ldsc()` function requires:
1.   `traits`: A vector of file paths pointing to the munged sumstats.
2.   `sample.prev`: Sample prevalence of the trait (use `NA` for continuous traits or if unknown).
3.   `population.prev`: Population prevalence of the trait (use `NA` for continuous traits or if unknown).
4.   `ld`: The directory containing the pre-computed LD scores.
5.   `wld`: The directory containing the LD scores used for weights (often the same as `ld`).
6.   `trait.names`: A character vector of trait names.


---
## 📁 3. Directory Structure & Data Preparation

The code below assumes you have a folder structured like this:

```text
LDSC_Practical_1/
  EUR/
    w_hm3.snplist
    eur_w_ld_chr/
    CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
    BMI.sumstats.gz
```


In [ ]:
%%R
# Set directories
sumstat_dir  <- "EUR/"
ref_data_dir <- "EUR/eur_w_ld_chr/"


### 🌍 Reference Data: HapMap 3 (HM3) SNPs
The reference directory includes the HapMap 3 (HM3) SNP IDs and their pre-computed LD scores.


In [ ]:
%%R
# Quick check
getwd()
dir(ref_data_dir)


**Why HapMap 3 SNPs?**
For LDSC to work as expected, the included SNPs must undergo rigorous Quality Control (QC). To ensure this, LDSC is typically performed using only HM3 SNPs because:
* HM3 SNPs are expected to be well-imputed across most GWAS cohorts.
* This reduces noise from poorly imputed or ultra-rare variants.


Let's inspect the HM3 SNP list that we will pass to `munge()`.


### ❓ Question 1: HapMap 3 SNPs
How many SNPs are there in the HapMap3 reference file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There are **1,217,311** SNPs included in the HM3 reference file.

*(Note: The first line is the header, so the true count is `wc -l` minus 1).*
</details>


In [ ]:
%%R
hm3_file <- file.path(ref_data_dir, "w_hm3.snplist")

# Take a look at the first few lines
cat(system(paste("head", hm3_file), intern = TRUE), sep = "\n")

# Count the number of lines
cat(system(paste("wc -l", hm3_file), intern = TRUE), sep = "\n")


We will restrict the LDSC analyses to the HM3 SNPs in this practical because we only have the pre-computed LD scores for these SNPs. Alternatively, one could compute LD scores for all QC-ed SNPs in a reference panel, but that requires more extensive computation!


### 🛡️ QC Filters for LDSC

We can also specify additional QC filters during munging.
For example, we might drop SNPs with imputation $R^2 < 0.90$ or $\text{MAF} < 0.01$.

*(Note: These filters will only be applied if the GWAS sumstats file includes `INFO` and `MAF` columns).*


In [ ]:
%%R
info_cutoff <- 0.9
maf_cutoff  <- 0.01


---
## 🧹 4. Munging Summary Statistics: BMI Example

We will perform the analysis in two steps. 
First, we use **`munge()`** to "clean" your GWAS sumstats file by selecting and renaming the required columns. It also filters out variants not present in the reference list, removes non-biallelic variants, and drops variants based on MAF and INFO cutoffs.

For this tutorial, we will practice `munge()` using only **Chromosome 22** data for Body Mass Index (BMI).


### Inspecting the Raw Data
LDSC requires specific column names in the munged data:
* `SNP` - SNP identifier (e.g., rsID).
* `N` - Sample size.
* `Z` - Z-score (or alternatively, a P-value and a signed statistic like `BETA` or `OR`).
* `A1` - Effect allele.
* `A2` - Other allele.


In [ ]:
%%R
bmi_file <- file.path(
  sumstat_dir,
  "CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz"
)
cat(system(paste0("zcat ", bmi_file, " | head"), intern=TRUE), sep="\n")


### 🔄 Reformatting the Sumstats
We need to verify which allele is the "effect allele" for which the association statistic `BETA` is signed (positive or negative).
Let's read the data, rename the columns to match what `munge()` expects, and save a clean copy.


In [ ]:
%%R
bmi_GWAS <- fread(bmi_file, header = TRUE, data.table = FALSE)
str(bmi_GWAS)
bmi_GWAS <- dplyr::rename(
  bmi_GWAS,
  A1 = Tested_Allele,
  A2 = Other_Allele
)
head(bmi_GWAS)


In [ ]:
%%R
bmi_updated_file <- file.path(sumstat_dir, "BMI_GWAS_for_LDSC.txt")
fwrite(
  bmi_GWAS,
  file = bmi_updated_file,
  sep = "\t",
  col.names = TRUE,
  row.names = TRUE,
  quote = FALSE
)
cat(system(paste("head", bmi_updated_file), intern=TRUE), sep="\n")


> ⚠️ **Note:** Some GWAS sumstats may report the "effect allele frequency" rather than MAF. In that case, we'd need to manually convert the effect allele frequency to MAF before filtering!


### 🏃‍♂️ Running `munge()`
Now, let's run the `munge()` function on our updated BMI sumstats.


In [ ]:
%%R
library(GenomicSEM)
bmi_munged <- file.path(sumstat_dir, "munged_BMI_GWAS_chr22_for_LDSC")

munge(
  files = bmi_updated_file,
  hm3 = hm3_file,
  trait.names = bmi_munged,
  info.filter = info_cutoff,
  maf.filter = maf_cutoff
)


### 🔍 Reviewing the Munge Log
Examine the screen output/log from the `munge()` command and answer the following questions:


### ❓ Question 2: Effect Allele
Which column is interpreted as the effect allele (A1)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The **`A1`** column!

Remember that we explicitly renamed `Tested_Allele` to `A1` to ensure `GenomicSEM` correctly identified it.
</details>


### ❓ Question 3: Starting SNPs
How many SNPs were present in the raw sumstats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There were **29,991 SNPs** present in the raw sumstats file.
</details>


### ❓ Question 4: Filtered by Reference
How many SNPs were dropped because they were not present in the reference HapMap3 file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**15,608 SNPs** were dropped because they were not in the HM3 reference list.
</details>


### ❓ Question 5: Filtered by INFO
How many SNPs were dropped due to low imputation quality (INFO)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**None!** There was no `INFO` column in the sumstats, so this filter was not applied.
</details>


### ❓ Question 6: Filtered by MAF
How many SNPs were dropped due to low or missing MAF?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**1 SNP** was dropped due to low or missing MAF.

*(Usually, there will be more, but this file was already pre-filtered prior to the tutorial).*
</details>


### ❓ Question 7: Final SNPs
How many SNPs are left in the munged sumstats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**14,382 SNPs** are left in the final munged sumstats file.
</details>


---
## 🧮 5. Running LDSC Heritability Analysis

We are done munging the Chromosome 22 subset! 

For the actual heritability analysis, we need genome-wide data. The full BMI summary statistics have already been munged for you (for all chromosomes). Let's quickly compare the raw file with the munged file.


In [ ]:
%%R
raw_bmi    <- file.path(sumstat_dir, "Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz")
munged_bmi <- file.path(sumstat_dir, "Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz")

cat(system(paste0("zcat ", raw_bmi, " | head"), intern=TRUE), sep='\n')


Let's look at the first few lines of the munged sumstats.


### ❓ Question 8: Munged Columns
What are the columns included in the munged sumstats?

<details>
<summary>💡 Click here to reveal the answer!</summary>

`SNP`, `N`, `Z`, `A1`, `A2`
</details>


### 🚀 Executing `ldsc()`
Now, let's run the LDSC analysis on BMI using the full, genome-wide munged sumstats.


In [ ]:
%%R
bmi_ldsc_out <- file.path(sumstat_dir, "BMI")

ldsc(
  traits = munged_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_ldsc_out
)


> ⚠️ **Note:** The `ldsc()` function in `GenomicSEM` is primarily designed for *bivariate* LDSC (genetic correlation), so it may print a warning about needing two or more traits. You can safely ignore that warning for this univariate heritability practical!


### 📊 Interpreting LDSC Output

### ❓ Question 9: Inflation Metrics
What are the Mean $\chi^2$ and the $\lambda_{\text{GC}}$? Do these suggest an inflation of GWAS statistics?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Mean $\chi^2$** = 3.9345
* **$\lambda_{\text{GC}}$** = 4.2057

Yes! These values are much greater than 1, indicating massive inflation in the test statistics. The question is: is this inflation due to true polygenicity, or confounding (like population stratification)?
</details>


### ❓ Question 10: Intercept
What is the estimate of the LDSC intercept in this analysis? Does it suggest significant bias in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Intercept** = 1.0592 (SE 0.0213)

While the intercept is slightly $> 1$, the ratio of the intercept is `0.0202` (SE `0.0073`). This tells us that only ~2% of the massive inflation we observed is due to confounding biases. The vast majority of the inflation is due to true, highly polygenic biological signal!
</details>


### ❓ Question 11: Heritability
What is the estimated SNP heritability ($h^2_{\text{SNP}}$) of BMI?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**$h^2_{\text{SNP}}$ of BMI = 0.2091** (SE 0.0063)

*(Or roughly ~21% of the variance in BMI is explained by these common SNPs).*
</details>


🎉 **Congratulations!** You successfully ran an LDSC analysis of BMI.

*Note: LDSC uses the munged sumstats to estimate the heritability/variance per SNP. The program then uses the estimated number of independent SNPs in the reference panel to extrapolate the total SNP heritability.*


---
## 🌟 6. Bonus (Time Permitting): LDSC with LMM Sumstats

We will now compare our previous results to an LDSC analysis using BMI GWAS sumstats from the **Pan-UKBB project**. 

These analyses were performed using a **Linear Mixed Model (LMM)** (specifically SAIGE), which inherently controls for population stratification and relatedness during the association testing phase.

Let's run LDSC on these LMM-derived summary statistics!


In [ ]:
%%R
munged_lmm_bmi <- file.path(sumstat_dir, "BMI.sumstats.gz")

cat(system(paste0("zcat ", munged_lmm_bmi, " | head"), intern=TRUE), sep='\n')

bmi_lmm_ldsc_out <- file.path(sumstat_dir, "BMI_LMM")

ldsc(
  traits = munged_lmm_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_lmm_ldsc_out
)


### ❓ Question 12: LMM Intercept
What is the estimate of the LDSC intercept? Does it suggest significant bias in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Intercept** = 0.8143 (SE 0.0163)

The LDSC intercept is *less than 1*! This is unusual. When the intercept is $< 1$, the ratio calculation yields a negative value (and it may be printed as `NA` or drop out). 

This happens because the Linear Mixed Model already perfectly (or even overly) corrected for population structure and relatedness!
</details>


### 🤔 Discussion on $N_{\text{eff}}$
When looking at LMM results, we may want to consider some additional points:

The **Effective $N$** (sometimes referred to as $N_{\text{eff}}$) is often less than the raw sample size $N$ due to the relatedness between some participants being accounted for by the mixed model.

*Could this artificially deflate the intercept? What does this mean for our heritability estimate? Discuss this with your instructors!*


---
## 🏁 End of Practical 1

You have now completed the Colab version of Practical 1 for SNP heritability using LDSC!
